In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt

First, let's look at a typical optimization problem using PyTorch and the autograd engine. We'll work with a very simple problem, finding the minimum of `y=(x-1)^2`, which we already know is 1.

In [ ]:
def myfn(xs):
    # y = x^2 - 2x + 1
    # dy/dx = 2x - 2
    return (xs-1)**2

# Our initial guess is x = 10, and initial gradient is 18.
x = torch.tensor([10], dtype=torch.float32, requires_grad=True)

In [ ]:
stepsz = 0.1

# Run one calculation of our quadratic function
y = myfn(x)
# backward() calculates the derivative using back propagation
# After calling backward, x.grad is set to dy/dx
y.backward()
print("At {}, the slope is {}".format(x.item(), x.grad.item()))

# Plot our quadratic function, and the current guess of the minimum
plotx = torch.arange(-15,16) # Left inclusive, right exclusive
ploty = myfn(plotx)
plt.plot(plotx, ploty)
plt.plot(x.detach(), y.detach(), 'ro')

# Step the parameter x by stepsz * the gradient. In this case we need to
# step AGAINST the gradient because we're looking for the minima.
x.data = x - stepsz * x.grad
print("Stepped x by {}, now {}".format(stepsz * x.grad.item(), x.item()))
x.grad = None
# Plot the new estimate after the step.
plt.plot(x.detach(), myfn(x.detach()), 'go')

# Run this cell as many times as you want; it will eventually make it to 1.
# Try again 

Next, we'll define a very simple custom function that we'll try to fit with a simple neural network. I chose a quadratic function, but it's worth coming back to this later and trying something more interesting, such as a transcendental function.

In [ ]:
def qfunction(x: float) -> float:
    return 0.14 * x**2 - 0.57 * x + 1.2

Let's just see what this function looks like briefly.

In [ ]:
xs = torch.linspace(-15, 15, 100)
ys = qfunction(xs)
plt.plot(xs, ys)

Define a simple model that will try to fit our function. This model is simply Linear -> ReLU -> Linear.
ReLU is a specific non-linear function that works like this
`y = { 0 if x < 0, x if x > 0 }`

Any sequence of ONLY linear layers can be condensed into a single (probably larger) linear layer. Linear layers can each be written as a matrix multiplication, and any sequence of matrix multiplications can be condensed into one. Adding ReLU makes this no longer true; ReLU is non-linear and cannot be expressed by multiplication and addition alone. Non-linear functions (which you'll see referred to as activations) are added to networks so that they can approximate other non-linear functions.

In [ ]:
# <EDIT THIS by adding more layers>
class MyNn(nn.Module):
    def __init__(self, features: int):
        super().__init__()
        self.lin1 = nn.Linear(1, features)
        # self.inner = nn.Linear(features, features)
        self.lin2 = nn.Linear(features, 1)

    def forward(self, x):
        x = self.lin1(x)
        x = F.relu(x)
        # x = self.inner(x)
        # x = F.relu(x)
        x = self.lin2(x)
        return x
        

In [ ]:
# Features defines how "big" the network is, that is, how many internal weights it can leverage.
# <EDIT THIS>
model = MyNn(features = 4)

# L1 Loss is calculated as the mean of the absolute difference between the prediction and label.
# So simply, l1loss = mean(|label - pred|)
def l1loss(pred, label):
    diff = (pred - label).abs()
    return diff.mean()

# We define a batchsize of 8. An entire batch of samples is run through the model on each iteration
batchsize = 8

# We'll train by sampling our custom function 1000 times between -5 and 5.
s_num = []
losses = []

# Learning rate affects the magnitude of each optimization step
# Try modifying this loop to reduce the learning rate by a small amount over time.
# <EDIT THIS>
lr = 0.1

# <EDIT THIS>
for x in range(1000):
    # rand returns between 0 and 1, so the shift and scale
    # sets the range to be between -15 and 15.
    input = (torch.rand(batchsize, 1) - 0.5) * 30
    label = qfunction(input)

    pred = model(input)
    loss = l1loss(pred, label)
    loss.backward()

    # Most basic possible optimizer: Step every parameter down by the gradient * LR
    for p in model.parameters():
        p.data = p - p.grad * lr
        p.grad = None

    s_num.append(x)
    losses.append(loss.item())
    # <EDIT THIS>
    lr = lr * 1.0
    
print("Final loss of {}".format(loss))
print(lr)
plt.plot(s_num, losses)

In [ ]:
model.eval()
with torch.no_grad():
    ps = model(xs.unsqueeze(-1))
    print(ps.shape)

plt.plot(xs, ys, label='Ground Truth')
plt.plot(xs, ps.flatten(), label='Prediction')
plt.legend()

Key topics from this notebook
* Linear Layers
* Activation functions such as ReLU
* Optimizers like Stochastic Gradient Descent
* Loss functions like L1 Loss
* "Hyper Parameters"
  * Batchsize
  * Learning Rate

As homework...
* The default configuration doesn't fit the function terribly well. Try altering the model and training parameters to fit the data better without blowing up the time it takes to train.
  * To accomplish this, you can change the learning rate, linear layers' feature size, batchsize, number of samples to train on, or even the optimizer type and loss function.
* Replace the quadratic function with one of your own, and try to fit both functions with the same hyperparameters. Was your function "harder" or "easier" than the quadratic function?